# OceanBench near-real-time evaluation

This notebook demonstrates the near-real-time (NRT) evaluation path introduced on branch `272`. It evaluates one recent GLONET forecast initialization against the most recent available Class IV observations.

NRT evaluation is intentionally different from the annual benchmark: it is a daily monitoring and scientific-evaluation workflow, with no historical ranking and no GLO12/GLORYS reference comparison.

## Running this notebook from branch 272

The NRT API is branch-specific and may not be available in the released package installed from PyPI. In an EDITO Jupyter service, the recommended setup is to configure the Git source as `https://github.com/mercator-ocean/oceanbench` and select branch `272-Enable-latest-forecasts-evaluation-against-observations`.

The first code cell supports both a service that already pulled that repository and a clean Jupyter workspace: if no checkout is found, it installs the package and its dependencies directly from that Git branch. The EDITO kernel must allow package installation and use Python 3.12.9 or newer.

In [ ]:
from pathlib import Path
import subprocess
import sys

oceanbench_branch = "272-Enable-latest-forecasts-evaluation-against-observations"
oceanbench_repository_url = "https://github.com/mercator-ocean/oceanbench.git"
repository_root = None
for candidate in (Path.cwd(), *Path.cwd().parents):
    if (candidate / "oceanbench").is_dir():
        repository_root = candidate
        break

if repository_root is None:
    if sys.version_info < (3, 12, 9):
        raise RuntimeError("This OceanBench branch requires Python 3.12.9 or newer.")
    subprocess.check_call(
        [
            sys.executable,
            "-m",
            "pip",
            "install",
            "--quiet",
            f"git+{oceanbench_repository_url}@{oceanbench_branch}",
        ]
    )
else:
    if str(repository_root) not in sys.path:
        sys.path.insert(0, str(repository_root))

import oceanbench
import pandas as pd
import xarray as xr
from IPython.display import display

print(f"Repository: {repository_root}")
print(f"OceanBench version: {oceanbench.__version__}")

## Select a fully evaluable forecast

The live observation bucket publishes an availability manifest. We use that manifest instead of a date hard-coded in the branch, because the retained observation window moves every day. We then verify the forecast objects themselves, since a date folder can exist before its Zarr data are published.

The cutoff and URLs can be overridden through the branch's environment variables when the live data source moves.

In [ ]:
import os
import requests

from oceanbench.core.dataset_utils import LEAD_DAYS_COUNT

availability_url = (
    "https://s3.waw3-1.cloudferro.com/"
    "oceanbench-bucket/public/live_observations/availability.json"
)
availability = requests.get(availability_url, timeout=30).json()
observation_cutoff = pd.Timestamp(availability["latest_complete_observation_day"])
forecast_init = pd.Timestamp(availability["latest_evaluable_forecast_init"])
observation_zarr_template = availability["observation_template"]
forecast_zarr_template = os.environ.get(
    "OCEANBENCH_LIVE_GLONET_FORECAST_ZARR_TEMPLATE",
    "https://s3.waw3-1.cloudferro.com/moiai-octo-bucket/"
    "public/octo/v0/ai-gallery/octo-glonet-p1d/{date}/{date}.zarr",
)
def _forecast_url_for(first_day_datetime):
    return forecast_zarr_template.format(
        compact_date=first_day_datetime.strftime("%Y%m%d"),
        day=first_day_datetime.strftime("%Y%m%d"),
        date=first_day_datetime.strftime("%Y-%m-%d"),
        yyyymmdd=first_day_datetime.strftime("%Y%m%d"),
        YYYYMMDD=first_day_datetime.strftime("%Y%m%d"),
    )

def _forecast_metadata_status_for(first_day_datetime):
    forecast_url = _forecast_url_for(first_day_datetime)
    return forecast_url, {
        metadata_name: requests.get(
            f"{forecast_url.rstrip(chr(47))}/{metadata_name}", timeout=30
        ).status_code
        for metadata_name in (".zmetadata", ".zgroup")
    }

candidate_forecast_inits = pd.date_range(
    forecast_init,
    observation_cutoff - pd.Timedelta(days=1),
    freq="D",
)[::-1]
available_forecast_inits = []
for candidate_forecast_init in candidate_forecast_inits:
    candidate_forecast_url, candidate_metadata_status = _forecast_metadata_status_for(
        candidate_forecast_init
    )
    if any(status == 200 for status in candidate_metadata_status.values()):
        available_forecast_inits.append(candidate_forecast_init)

if not available_forecast_inits:
    raise FileNotFoundError(
        f"No populated forecast store was found between {forecast_init.date()} and "
        f"{observation_cutoff.date()} before the observation cutoff."
    )

forecast_init = available_forecast_inits[0]
forecast_url, forecast_metadata_status = _forecast_metadata_status_for(forecast_init)
if not any(status == 200 for status in forecast_metadata_status.values()):
    raise FileNotFoundError(
        f"No forecast store found at {forecast_url}. "
        f"Metadata status: {forecast_metadata_status}. "
        "Set OCEANBENCH_LIVE_GLONET_FORECAST_ZARR_TEMPLATE to the current producer URL."
    )

print(f"Observation cutoff: {observation_cutoff.date()}")
print(f"Forecast initialization: {forecast_init.date()}")
forecast_lead_days = availability["forecast_lead_days"]
print(f"Forecast lead-time horizon: {forecast_lead_days} days")
print(f"Observation source: {observation_zarr_template}")
print(f"Forecast source: {forecast_url}")

## Open the live GLONET forecast

`glonet_latest` opens one forecast initialization and normalizes it to OceanBench's standard `first_day_datetime` and `lead_day_index` dimensions. The data remain lazy until a diagnostic needs to read them.

In [ ]:
challenger_dataset = oceanbench.datasets.challenger.glonet_latest(
    first_day_datetime=forecast_init.to_pydatetime(),
    zarr_template=forecast_zarr_template,
)

display(challenger_dataset)

In [ ]:
dimension_summary = pd.DataFrame(
    {"size": {dimension: int(size) for dimension, size in challenger_dataset.sizes.items()}}
)
variable_summary = pd.DataFrame(
    {"dimensions": {variable: tuple(data_array.dims) for variable, data_array in challenger_dataset.data_vars.items()}}
)

display(dimension_summary)
display(variable_summary)
display(pd.Series(challenger_dataset.attrs, name="value").to_frame())

## Prepare the NRT evaluation report

The live report is Class-IV-only. It loads the recent observations lazily and exposes two families of diagnostics:

- variable errors and RMSD for temperature, salinity, sea-level anomaly and currents;
- Lagrangian trajectory deviation for drifter observations.

Evaluation is regional, so the same report object can be created for the global domain or for the official IBI region.

In [ ]:
from oceanbench.core.evaluation_report import prepare_live_evaluation_report

region = "global"
evaluation_report = prepare_live_evaluation_report(
    challenger_dataset,
    region=region,
    observation_zarr_template=observation_zarr_template,
    observation_last_available_day=observation_cutoff.strftime("%Y-%m-%d"),
)

print(f"Prepared NRT report for region: {region}")

## Inspect the matched Class IV observations

The observation selection is based on the forecast initialization and lead-time coordinates. Only observations falling in the forecast window are retained.

In [ ]:
class4_observation_report = evaluation_report.class4_observation
observation_dataset = class4_observation_report.dataset

if observation_dataset is None:
    display(class4_observation_report.rmsd)
else:
    display(observation_dataset)
    print(f"Matched observations: {observation_dataset.sizes.get('observations', 0)}")

## Variable scores against recent observations

The returned table contains the lead-time and variable/depth breakdown used by the NRT report. Accessing `.rmsd` triggers the required computation.

In [ ]:
display(evaluation_report.class4_observation.rmsd)

## Drifter trajectory deviation

This diagnostic compares the forecast-advected particles with Class-IV drifter tracks at the drogue depth. The branch keeps tracks alive across sparse hourly frames and reports masked deviations when a comparison cannot be made.

In [ ]:
display(evaluation_report.class4_drifter_trajectory_deviation)

## Interactive diagnostic explorers

The next two properties return HTML explorers suitable for a Jupyter output cell. They are also embedded in the generated live evaluation report notebooks used by the website.

In [ ]:
evaluation_report.class4_observation_error_explorer

In [ ]:
evaluation_report.class4_drifter_trajectory_explorer

## Re-run the same forecast for the IBI region

NRT evaluation supports the official IBI region as well as the global domain. This reuses the already opened forecast and applies the regional spatial subset before matching observations.

In [ ]:
ibi_evaluation_report = prepare_live_evaluation_report(
    challenger_dataset,
    region="ibi",
    observation_zarr_template=observation_zarr_template,
    observation_last_available_day=observation_cutoff.strftime("%Y-%m-%d"),
)

display(ibi_evaluation_report.class4_observation.rmsd)

## From a demo notebook to the daily NRT job

The production orchestration is implemented by `validate_nrt_forecast`. It waits for the forecast success marker, runs a generated live report notebook, optionally cleans up a temporary forecast, and writes or merges the NRT manifest consumed by the website.

The call below is intentionally commented out: it can wait for remote data, write a manifest, publish a report, and delete a temporary forecast. Use it only with the intended credentials and output destination.

In [ ]:
from oceanbench.core.nrt_validation import forecast_init_for_observation_cutoff

print(
    forecast_init_for_observation_cutoff(
        observation_cutoff.strftime("%Y-%m-%d")
    )
)

# Production-only example; keep commented during a local demonstration.
# from oceanbench.core.nrt_validation import validate_nrt_forecast
# result, manifest_path = validate_nrt_forecast(
#     system_id="octo-glonet-p1d",
#     system_label="GLONET",
#     forecast_init=forecast_init.strftime("%Y-%m-%d"),
#     observation_cutoff=observation_cutoff.strftime("%Y-%m-%d"),
#     manifest_path="nrt-validation-manifest.json",
# )